# Build a Bedrock Knowledge Base
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [1]:

# Tools for building and populating Vertex AI data stores and search apps
import bedrock_kb_security_build as bkbsb # Security
import bedrock_kb_build as bkbb # Vectorized knowledge base

# AWS Python
import boto3
from opensearchpy import AWSV4SignerAuth


## Set up configuration and initialize

In [2]:
# --- Configuration ---
REGION_NAME = 'us-east-2'
BUCKET_NAME = 'rag-search-tests'
S3_PREFIX = 'documents/'
COLLECTION_NAME = 'rag-kb-collection'
KB_NAME = 'rag-search-kb'
INDEX_NAME = 'bedrock-knowledge-base-default-index'
ROLE_NAME = 'AmazonBedrockExecutionRoleForKnowledgeBase'
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'

# Initialize AWS clients
admin_session = boto3.Session(profile_name='ns-admin',
                              region_name=REGION_NAME)
aoss_client = admin_session.client('opensearchserverless', region_name=REGION_NAME)
iam_client = admin_session.client('iam')
sts_client = admin_session.client('sts')
bedrock_agent = admin_session.client('bedrock-agent', region_name=REGION_NAME)

# Get user information
account_id = sts_client.get_caller_identity()['Account']
current_user_arn = sts_client.get_caller_identity()['Arn']


## Create Security Policies

In [4]:
# Create or verify network policy
bkbsb.ensure_aoss_policy(
    name=f'{COLLECTION_NAME}-enc',
    client=aoss_client,
    policy_type='encryption',
    policy_doc={
        "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
        "AWSOwnedKey": True
    }
)

# Create or verify network policy
bkbsb.ensure_aoss_policy(
    name=f'{COLLECTION_NAME}-net',
    client=aoss_client,
    policy_type='network',
    policy_doc=[{
        "Rules": [
            {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
            {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}
        ],
        "AllowFromPublic": True
    }]
)

# Create or verify IAM execution role with Bedrock and S3 privileges
role_arn = bkbsb.ensure_execution_role(client=iam_client,
                                       role_name=ROLE_NAME,
                                       bucket_name=BUCKET_NAME,
                                       region_name=REGION_NAME,
                                       embedding_model=EMBEDDING_MODEL)

# Ensure data policy exists for knowledge base
bkbsb.ensure_data_access_policy(role_arn=role_arn,
                                client=aoss_client,
                                collection_name=COLLECTION_NAME,
                                account_id=account_id,
                                current_user_arn=current_user_arn)



Encryption policy already exists: rag-kb-collection-enc
Network policy already exists: rag-kb-collection-net
Role already exists: AmazonBedrockExecutionRoleForKnowledgeBase
Upserted inline role policy: BedrockPolicy
Data access policy already up to date: rag-kb-collection-acc


## Check permissions

In [5]:
# Check user permissions
bkbsb.ensure_iam_role(role_arn=role_arn,
                      client=iam_client)

Using role: arn:aws:iam::584560776394:role/AmazonBedrockExecutionRoleForKnowledgeBase
✓ Role exists


## Add security roles

In [6]:
# Ensure policy propagation before continuing
bkbsb.ensure_policy_propagation(role_arn=role_arn,
                                iam_client=iam_client,
                                aoss_client=aoss_client,
                                role_name=ROLE_NAME,
                                collection_name=COLLECTION_NAME)


Using role: arn:aws:iam::584560776394:role/AmazonBedrockExecutionRoleForKnowledgeBase
✓ Access policy visible (version MTc3NjM4NDEwMzg1N18y); proceeding.


## Create vector index

In [7]:
# Set up OpenSearch client with AWS auth using admin_session
credentials = admin_session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, REGION_NAME, 'aoss')

bkbb.create_vector_index(index_name=INDEX_NAME,
                         aoss_client=aoss_client,
                         awsauth=awsauth,
                         collection_name=COLLECTION_NAME,
                         delete_existing=True
                        )


✓ Vector index already exists: bedrock-knowledge-base-default-index
✓ Deleting bedrock-knowledge-base-default-index and creating a new vector index: bedrock-knowledge-base-default-index


## Set up Knowledge Base

In [8]:
# Delete the existing knowledge base
bkbb.delete_existing_kb(kb_name=KB_NAME,
                        bedrock_agent=bedrock_agent)


Checking for existing Knowledge Base named 'rag-search-kb'...
Found existing KB RJEZ40U7RT. Deleting...
Waiting for KB deletion to complete...
✓ Existing KB deleted


In [9]:
# Create a knowledge base
kb_id = bkbb.create_kb(kb_name=KB_NAME,
                       bedrock_agent=bedrock_agent,
                       aoss_client=aoss_client,
                       role_arn=role_arn,
                       collection_name=COLLECTION_NAME,
                       index_name=INDEX_NAME,
                       region_name=REGION_NAME,
                       embedding_model=EMBEDDING_MODEL
                       )


✓ Knowledge Base Created: IKMZTZONOU


## Ingest documents to the Knowledge Base

In [10]:
# Ingest documents
ingestion_response = bkbb.ingest_docs_kb(kb_id=kb_id,
                                         bedrock_agent=bedrock_agent,
                                         bucket_name=BUCKET_NAME,
                                         s3_prefix=S3_PREFIX
                                         )


Created data source: VXJBQNRIDD
✓ Ingestion started for rag-search-tests/documents/
You can now filter by 'source_type' or 'doc_index' in your queries!


## Check status of ingestion job

In [13]:
ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']
ds_id = ingestion_response["ingestionJob"]["dataSourceId"]
print(f"Monitoring Ingestion Job: {ingestion_job_id}...")

bkbb.check_status_ingestion_job(ingestion_job_id=ingestion_job_id,
                                bedrock_agent=bedrock_agent,
                                kb_id=kb_id,
                                ds_id=ds_id
                                )


Monitoring Ingestion Job: R4W15DKWLA...
Current Status: COMPLETE

INGESTION JOB SUMMARY
Status: COMPLETE
Documents Scanned: 20
Documents Indexed: 20
Documents Failed:  0

Full JSON Job Details (for debugging):
{
  "knowledgeBaseId": "IKMZTZONOU",
  "dataSourceId": "VXJBQNRIDD",
  "ingestionJobId": "R4W15DKWLA",
  "status": "COMPLETE",
  "statistics": {
    "numberOfDocumentsScanned": 20,
    "numberOfMetadataDocumentsScanned": 20,
    "numberOfNewDocumentsIndexed": 20,
    "numberOfModifiedDocumentsIndexed": 0,
    "numberOfMetadataDocumentsModified": 0,
    "numberOfDocumentsDeleted": 0,
    "numberOfDocumentsFailed": 0
  },
  "startedAt": "2026-05-05 20:11:09.028771+00:00",
  "updatedAt": "2026-05-05 20:11:32.895203+00:00"
}
